# Lakehouse Agent - Deploy Athena Database

This notebook deploys the Athena database and tables for the lakehouse data layer.

**What this notebook does:**
- Creates S3 bucket for data storage (if needed)
- Uploads sample claims and users data to S3
- Creates Athena database: `lakehouse_db`
- Creates tables: `claims` and `users`
- Verifies deployment with test queries

**Prerequisites:**
- ✅ Completed `00-prerequisites-setup.ipynb`
- ✅ SSM parameters configured with `lh_` prefix
- ✅ AWS credentials with Athena, S3, and Glue permissions

**IAM Permissions Required:**
- `athena:*`
- `s3:*`
- `glue:*`
- `ssm:GetParameter`

**Duration:** ~15 minutes

In [ ]:
import boto3
import subprocess
import sys
from pathlib import Path

# Add parent directory to import config
sys.path.insert(0, str(Path.cwd()))

from config import config

print("✅ Imports successful")

## Step 1: Validate Prerequisites

Check that all required SSM parameters from the previous notebook exist.

In [ ]:
# Initialize AWS clients
ssm_client = boto3.client('ssm')

# Check required parameters
print("🔍 Validating prerequisites...\n")

required_params = [
    'lh_s3_bucket_name',
    'lh_athena_database_name',
    'lh_athena_workgroup'
]

missing = []
for param in required_params:
    try:
        response = ssm_client.get_parameter(Name=param)
        value = response['Parameter']['Value']
        print(f"✅ {param}: {value}")
    except ssm_client.exceptions.ParameterNotFound:
        print(f"❌ {param}: NOT FOUND")
        missing.append(param)

if missing:
    print(f"\n❌ Missing parameters: {', '.join(missing)}")
    print("Please run 00-prerequisites-setup.ipynb first")
else:
    print("\n✅ All prerequisites validated!")
    
    # Load configuration
    BUCKET_NAME = config.S3_BUCKET_NAME
    DATABASE_NAME = config.ATHENA_DATABASE_NAME
    WORKGROUP = config.ATHENA_WORKGROUP
    
    print(f"\n📋 Configuration:")
    print(f"   Bucket: {BUCKET_NAME}")
    print(f"   Database: {DATABASE_NAME}")
    print(f"   Workgroup: {WORKGROUP}")

## Step 2: Deploy Athena Database

Run the Athena setup script to create the database, tables, and upload sample data.

In [ ]:
print("🚀 Running Athena setup...\n")

result = subprocess.run(
    ['python', 'setup_athena_with_config.py'],
    cwd='athena-setup',
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("❌ Error during Athena setup:")
    print(result.stderr)
else:
    print("\n✅ Athena setup completed successfully!")

## Step 3: Validate Deployment

Verify that the database and tables were created successfully.

In [ ]:
print("🔍 Validating Athena deployment...\n")

athena_client = boto3.client('athena')
glue_client = boto3.client('glue')

# Check database exists
try:
    response = glue_client.get_database(Name=DATABASE_NAME)
    print(f"✅ Database '{DATABASE_NAME}' exists")
except glue_client.exceptions.EntityNotFoundException:
    print(f"❌ Database '{DATABASE_NAME}' not found")

# Check tables exist
try:
    response = glue_client.get_tables(DatabaseName=DATABASE_NAME)
    tables = response['TableList']
    
    print(f"\n📋 Tables in {DATABASE_NAME}:")
    for table in tables:
        table_name = table['Name']
        column_count = len(table['StorageDescriptor']['Columns'])
        print(f"   • {table_name} ({column_count} columns)")
    
    if len(tables) >= 2:
        print("\n✅ All tables created successfully")
    else:
        print(f"\n⚠️  Expected 2 tables, found {len(tables)}")
        
except Exception as e:
    print(f"❌ Error checking tables: {e}")

## Next Steps

✅ **Athena Database Deployment Complete!**

Your Athena database is now set up with:
- Database: `lakehouse_db`
- Tables: `claims` (9 sample claims), `users` (3 test users)
- S3 data location: `s3://{BUCKET_NAME}/lakehouse-data/`

**Next:** Run `02-deploy-lake-formation.ipynb` to set up row-level security.

### Test Queries

You can test the deployment with these queries in the Athena console:

```sql
-- Count all claims
SELECT COUNT(*) as total_claims FROM lakehouse_db.claims;

-- View claims for user001
SELECT claim_id, claim_type, claim_status, claim_amount 
FROM lakehouse_db.claims 
WHERE user_id = 'user001@example.com';

-- View all users
SELECT * FROM lakehouse_db.users;
```